#### LLM 토큰 스트리밍 (`messages`)
- `messages` 모드는 LLM의 응답을 토큰 단위로 실시간 스트리밍하는 강력한 기능
  
  **대화형 인터페이스에서 텍스트가 한 글자씩 나타나는 효과를 구현**
- `messages` 모드는 (`message_chunk`, `metadata`) 튜플 형태로 데이터를 반환

In [1]:
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessageChunk
from typing_extensions import TypedDict

In [5]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

True

In [6]:
class ChatState(TypedDict):
    messages: list

In [13]:
graph = StateGraph(ChatState)

- 노드 함수 정의

In [14]:
def llm_node(state: ChatState):
    llm = init_chat_model("openai:gpt-4.1-mini")
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

- 그래프 구성 및 컴파일

In [15]:
graph.add_node("llm", llm_node)
graph.add_edge(START, "llm")
graph.add_edge("llm", END)

app = graph.compile()

- 실행
  - `messages` 스트림에는 `AIMessageChunk` 외에도 `ToolMessage`, `HumanMessage` 등 다양한 메시지 타입이 포함될 수 있음
  - **`isinstance(chunk, AIMessageChunk)` 체크를 통해 실제 LLM 응답 토큰만 필터링하는 것이 공식 문서에서 권장하는 패턴**

In [16]:
for chunk in app.stream(
    {"messages": [HumanMessage(content="LangGraph란 무엇인가요?")]},
    stream_mode="messages"
):
    message_chunk, metadata = chunk
    if isinstance(message_chunk, AIMessageChunk) and message_chunk.content:
        print(message_chunk.content, end="", flush=True)

LangGraph는 자연어 처리(NLP)와 관련된 기술 또는 도구의 이름일 가능성이 높지만, 2024년 6월까지의 자료를 기준으로 특정한 "LangGraph"라는 용어에 대한 널리 알려진 정의나 제품, 프레임워크는 존재하지 않습니다.

다만, 이 단어를 구성하는 요소들을 통해 유추해볼 수 있습니다:

- **Lang**: 일반적으로 "Language"(언어)를 줄인 말로, 자연어 처리, 프로그래밍 언어 등 언어와 관련된 의미를 갖습니다.
- **Graph**: 그래프 구조, 즉 노드와 간선으로 이루어진 데이터 구조나 네트워크를 의미합니다.

따라서 LangGraph는 언어 데이터나 텍스트 데이터를 그래프 형태로 표현하거나, 언어 관련 문제를 그래프 이론적 방법으로 다루는 도구, 프레임워크 또는 개념체계일 가능성이 큽니다. 예를 들어, 문장 내 단어들 간의 관계를 그래프로 모델링하거나, 지식 그래프(Knowledge Graph)를 언어 이해에 적용하는 분야와 관련 있을 수 있습니다.

만약 특정 회사나 연구팀에서 "LangGraph"라는 이름으로 개발한 제품이나 프로젝트가 있다면, 더 구체적인 맥락이나 해당 출처의 정보를 알려주시면 더 자세하게 설명드릴 수 있습니다.

추가 질문이 있으면 말씀해 주세요!

- **메타데이터 활용**
  - 메타데이터에는 노드 이름, LLM 정보, 토큰 위치 등이 포함

In [ ]:
for chunk in app.stream(
    {"messages": [HumanMessage(content="안녕하세요")]},
    stream_mode="messages"
):
    message_chunk, metadata = chunk
    print(f"노드: {metadata.get('langgraph_node')}")
    print(f"토큰: {message_chunk.content}")
    print(f"메타데이터: {metadata}")
    print("-" * 50)

- **노드별 필터링**
  - 특정 노드의 LLM 출력만 스트리밍
  - **`metadata["langgraph_node"]`를 사용하여 원하는 노드의 출력만 필터링**

In [19]:
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage, AIMessageChunk
from typing_extensions import TypedDict

In [20]:
class AgentState(TypedDict):
    messages: list

graph = StateGraph(AgentState)

In [21]:
def analyzer_node(state: AgentState):
    llm = init_chat_model("openai:gpt-4.1-mini")
    system_msg = SystemMessage(content="당신은 텍스트 분석 전문가입니다.")
    response = llm.invoke([system_msg] + state["messages"])
    return {"messages": [response]}

def summarizer_node(state: AgentState):
    llm = init_chat_model("openai:gpt-4.1-mini")
    system_msg = SystemMessage(content="당신은 요약 전문가입니다.")
    response = llm.invoke([system_msg] + state["messages"])
    return {"messages": [response]}

In [22]:
graph.add_node("analyzer", analyzer_node)
graph.add_node("summarizer", summarizer_node)

graph.add_edge(START, "analyzer")
graph.add_edge("analyzer", "summarizer")
graph.add_edge("summarizer", END)

app = graph.compile()

In [25]:
print("[요약] ", end="", flush=True)

for chunk in app.stream(
    {"messages": [HumanMessage(content="LangGraph의 장점은?")]},
    stream_mode="messages"
):
    message_chunk, metadata = chunk
    node_name = metadata.get("langgraph_node")

    if node_name == "summarizer" and isinstance(message_chunk, AIMessageChunk) and message_chunk.content:
        print(message_chunk.content, end="", flush=True)

print()

[요약] LangGraph의 주요 장점은 다음과 같습니다:

- 자연어 데이터를 구조화된 그래프 형태로 표현하여 개체와 관계를 명확하게 파악할 수 있다.
- 복잡한 언어 의미와 맥락을 통합해 정보의 상호 연결성을 높인다.
- 확장성과 유연성이 뛰어나 새로운 개체와 관계를 쉽게 추가할 수 있다.
- 그래프 기반 질의 응답으로 보다 정확하고 심층적인 답변이 가능하다.
- 개념과 관계 기반 검색을 지원해 정보 탐색 효율을 향상시킨다.
- 지식 관리, 추천 시스템, 챗봇 등 다양한 분야에 효과적으로 적용된다.


<br>

- **태그 기반 필터링**
  - LLM 호출에 태그를 연결하여 특정 LLM의 출력만 스트리밍

In [26]:
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END
from langchain.messages import HumanMessage, AIMessageChunk
from typing_extensions import TypedDict

In [27]:
class MultiLLMState(TypedDict):
    messages: list
    analysis: str
    summary: str

graph = StateGraph(MultiLLMState)

In [28]:
def analyze_with_gpt4(state: MultiLLMState):
    llm = init_chat_model("openai:gpt-4.1", tags=["gpt4"])
    response = llm.invoke(state["messages"])
    return {"analysis": response.content}

def summarize_with_mini(state: MultiLLMState):
    llm = init_chat_model("openai:gpt-4.1-mini", tags=["mini"])
    response = llm.invoke(state["messages"])
    return {"summary": response.content}

In [29]:
graph.add_node("analyze", analyze_with_gpt4)
graph.add_node("summarize", summarize_with_mini)

graph.add_edge(START, "analyze")
graph.add_edge("analyze", "summarize")
graph.add_edge("summarize", END)

app = graph.compile()

In [30]:
print("=== GPT-4 출력만 스트리밍 ===")
for chunk in app.stream(
    {"messages": [HumanMessage(content="AI의 미래는?")]},
    stream_mode="messages"
):
    message_chunk, metadata = chunk
    if "gpt4" in metadata.get("tags", []) and isinstance(message_chunk, AIMessageChunk) and message_chunk.content:
        print(message_chunk.content, end="", flush=True)

=== GPT-4 출력만 스트리밍 ===
AI의 미래는 매우 밝고, 동시에 다양한 도전과 논쟁이 공존하는 복합적인 모습일 것으로 예상됩니다. 주요 흐름을 정리해 보면 다음과 같습니다:

1. **일상 속 AI 확산**  
   인공지능은 이미 스마트폰, 자동차, 스마트홈, 의료 등 다양한 분야에서 점차 넓고 깊게 활용되고 있습니다. 앞으로는 사물인터넷, 웨어러블 기기, 로봇 등 더 다양한 기기와 서비스에 AI가 내장되어 삶의 많은 부분이 자동화되고 개인화될 것입니다.

2. **더 발전된 생성 AI**  
   챗GPT, DALL-E, Midjourney 등 생성형 AI가 빠르게 발전하고 있습니다. 앞으로 텍스트뿐만 아니라 이미지, 영상, 음악 등 다양한 창작 영역에서 인간과 협업하거나 경쟁하게 될 가능성이 큽니다.

3. **초지능 & AGI(범용 인공지능)로의 접근**  
   AI 연구의 궁극적인 목표는 인간처럼 사고하고 학습할 수 있는 범용 인공지능(Artificial General Intelligence, AGI)입니다. 아직 갈 길이 멀지만, 점점 더 복잡하고 고차원적인 사고가 가능한 모델들이 등장할 것으로 기대됩니다.

4. **노동 시장의 변화**  
   단순 반복 업무는 빠르게 자동화될 것이고, 데이터 분석, 창의적 사고, 인간 중심의 서비스 등 일부 새로운 역할은 오히려 더 중요해질 수 있습니다. 인간-기계 협업이 핵심 역량이 됩니다.

5. **윤리와 규제의 중요성**  
   AI의 오용(편향, 차별, 사생활 침해 등)과 위험(가짜 정보, 딥페이크, 해킹 등)을 막기 위한 법적·윤리적 논의가 더욱 활발해질 것입니다. EU AI법, AI 투명성 등 규제 또한 더 다양해질 것으로 보입니다.

6. **인간 중심 AI로의 진화**  
   기술 발전과 함께, 인간의 가치와 권리를 중심에 두는 ‘책임 있는 AI’, ‘설명 가능한 AI’의 중요성이 커지고 있습니다.

**요약:**  
AI는 점점 더 우리의 삶에 깊게 스며들 것이며, 엄청난 기회

<br>

<hr>

<br>

#### 스트리밍 채팅 인터페이스 예시

In [31]:
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessageChunk
from typing_extensions import TypedDict

In [32]:
class ChatState(TypedDict):
    messages: list

In [33]:
def create_chat_app():
    graph = StateGraph(ChatState)

    def chat_node(state: ChatState):
        llm = init_chat_model("openai:gpt-4.1-mini")
        response = llm.invoke(state["messages"])
        return {"messages": [response]}

    graph.add_node("chat", chat_node)
    graph.add_edge(START, "chat")
    graph.add_edge("chat", END)

    return graph.compile()

In [34]:
app = create_chat_app()

In [35]:
def stream_chat_response(user_input: str):
    """사용자 입력에 대한 스트리밍 응답"""
    print("AI: ", end="", flush=True)

    full_response = ""
    for chunk in app.stream(
        {"messages": [HumanMessage(content=user_input)]},
        stream_mode="messages"
    ):
        message_chunk, _ = chunk
        if isinstance(message_chunk, AIMessageChunk) and message_chunk.content:
            token = message_chunk.content
            full_response += token
            print(token, end="", flush=True)

    print()
    return full_response

In [36]:
user_question = "LangGraph의 핵심 기능 3가지는?"
response = stream_chat_response(user_question)

AI: LangGraph의 핵심 기능 3가지는 다음과 같습니다:

1. **언어 데이터 그래프 구축**: 다양한 언어 데이터와 요소들을 노드와 엣지로 시각화하고 연관성을 체계적으로 표현하여 복잡한 언어 구조를 이해하기 쉽게 만듭니다.

2. **자연어 처리 통합**: 텍스트 분석, 의미 분석, 감성 분석 등 다양한 자연어 처리(NLP) 기술을 통합하여 그래프 데이터와 결합된 심층적인 언어 이해를 지원합니다.

3. **인터랙티브 시각화 및 탐색**: 사용자들이 그래프 내에서 데이터를 직관적으로 탐색하고 분석할 수 있도록 인터랙티브한 UI를 제공하여, 언어 데이터 기반 인사이트 도출을 용이하게 합니다.


<br>

#### 토큰 카운터 예시

In [37]:
def stream_with_token_count(user_input: str):
    """토큰 수를 세면서 스트리밍"""
    token_count = 0

    for chunk in app.stream(
        {"messages": [HumanMessage(content=user_input)]},
        stream_mode="messages"
    ):
        message_chunk, _ = chunk
        if isinstance(message_chunk, AIMessageChunk) and message_chunk.content:
            token = message_chunk.content
            token_count += 1
            print(token, end="", flush=True)

    print(f"\n\n총 토큰 수: {token_count}")

In [38]:
stream_with_token_count("간단히 설명해주세요")

네! 어떤 내용을 간단히 설명해드릴까요?

총 토큰 수: 13


<br>

#### 비동기 스트리밍
- 대규모 애플리케이션에서는 비동기 스트리밍을 사용

In [39]:
import asyncio
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessageChunk
from typing_extensions import TypedDict

In [40]:
class AsyncChatState(TypedDict):
    messages: list

In [41]:
def create_async_chat():
    graph = StateGraph(AsyncChatState)

    def chat_node(state: AsyncChatState):
        llm = init_chat_model("openai:gpt-4.1-mini")
        response = llm.invoke(state["messages"])
        return {"messages": [response]}

    graph.add_node("chat", chat_node)
    graph.add_edge(START, "chat")
    graph.add_edge("chat", END)

    return graph.compile()

In [42]:
async def async_stream_chat(user_input: str):
    app = create_async_chat()

    print("AI: ", end="", flush=True)
    async for chunk in app.astream(
        {"messages": [HumanMessage(content=user_input)]},
        stream_mode="messages"
    ):
        message_chunk, _ = chunk
        if isinstance(message_chunk, AIMessageChunk) and message_chunk.content:
            print(message_chunk.content, end="", flush=True)
    print()

In [44]:
# asyncio.run(async_stream_chat("안녕하세요"))

await async_stream_chat("안녕하세요")

AI: 안녕하세요! 무엇을 도와드릴까요?


<br>

#### 멀티 모델 스트리밍 예시

In [45]:
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessageChunk
from typing_extensions import TypedDict

In [46]:
class MultiModelState(TypedDict):
    messages: list

graph = StateGraph(MultiModelState)

- 노드 함수

In [47]:
def gpt_node_4(state: MultiModelState):
    llm = init_chat_model("openai:gpt-4.1-mini", tags=["gpt"])
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

def gpt_node_5(state: MultiModelState):
    llm = init_chat_model("openai:gpt-5.4-nano", tags=["gpt"])
    response = llm.invoke(state["messages"])
    return {"messages": [response]}


- 그래프 구성 및 컴파일

In [48]:
graph.add_node("gpt-4.1", gpt_node_4)
graph.add_node("gpt-5.4", gpt_node_5)

graph.add_edge(START, "gpt-4.1")
graph.add_edge("gpt-4.1", "gpt-5.4")
graph.add_edge("gpt-5.4", END)

app = graph.compile()

- 실행
  - 멀티 에이전트 시스템에서는 `metadata["langgraph_node"]`로 어느 에이전트(노드)가 생성한 청크인지 파악할 수 있음
  - 서브그래프를 사용하는 경우 `subgraphs=True` 옵션으로 `namespace`를 통해 어느 서브그래프에서 발생한 이벤트인지 구분할 수 있음

In [51]:
current_node = None

for chunk in app.stream(
    {"messages": [HumanMessage(content="AI란?")]},
    stream_mode="messages"
):
    message_chunk, metadata = chunk
    node_name = metadata.get("langgraph_node")

    if isinstance(message_chunk, AIMessageChunk) and message_chunk.content:
        if current_node != node_name:
            print(f"\n[{node_name}] ", end="", flush=True)
            current_node = node_name
        print(message_chunk.content, end="", flush=True)


[gpt-4.1] AI란 "인공지능"(Artificial Intelligence)의 약자로, 인간의 학습, 추론, 문제 해결, 언어 이해 등 지능적인 작업을 컴퓨터가 수행하도록 하는 기술을 말합니다. 즉, 기계가 인간처럼 사고하고 학습하며 스스로 판단할 수 있도록 만드는 기술과 방법론을 포함합니다.

AI의 주요 분야로는 머신러닝(기계학습), 딥러닝(심층학습), 자연어 처리, 컴퓨터 비전 등이 있으며, 음성 인식, 이미지 분석, 자율 주행 자동차, 챗봇 등 다양한 현실 세계의 응용 사례가 있습니다.
[gpt-5.4] AI는 **인공지능(Artificial Intelligence)** 의 약자로, **컴퓨터가 인간처럼 학습하고(데이터로부터), 추론하고(규칙/패턴으로 판단), 문제를 해결**하도록 만드는 기술을 말해요.  

쉽게 말하면, 사람이 하던 “생각”에 가까운 일을 **기계가 대신** 하게 만드는 분야입니다.

**대표 예시**
- **추천 시스템**(유튜브/넷플릭스/쇼핑 추천)
- **챗봇/번역**(자연어 처리)
- **얼굴 인식/사진 분석**(컴퓨터 비전)
- **음성 인식**(스마트폰 음성비서)
- **자율주행**(판단과 경로 예측)

**주요 종류(분야)**
- **머신러닝**: 데이터로 규칙을 학습  
- **딥러닝**: 신경망을 이용해 더 복잡한 학습  
- **자연어 처리**: 언어 이해/생성  
- **컴퓨터 비전**: 이미지/영상 이해  

원하시면 “AI랑 머신러닝/딥러닝 차이”도 짧게 정리해드릴까요?

<br>

<hr>

<br>

#### 커스텀 데이터 스트리밍 (`custom`)
- `custom` 모드는 노드 내부에서 사용자 정의 데이터를 자유롭게 스트리밍할 수 있는 기능
- 진행 상황, 로그, 중간 결과 등 그래프 상태나 LLM 출력이 아닌 데이터를 전송할 때 사용

<img src='img/1-11-9.png' width=600>

<br>

- **`get_stream_writer()`**
  - 스트림 작성자를 가져오고, `writer()` 메서드로 데이터를 전송
  - `get_stream_writer()`는 Python < 3.11 비동기 환경에서 동작하지 않음

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.config import get_stream_writer
from typing_extensions import TypedDict
import time

In [53]:
class ProcessState(TypedDict):
    data: list[int]
    result: int

graph = StateGraph(ProcessState)

In [54]:
def process_node(state: ProcessState):
    writer = get_stream_writer()

    writer("처리 시작...")

    total = 0
    for i, value in enumerate(state["data"]):
        total += value
        progress = (i + 1) / len(state["data"]) * 100
        writer(f"진행률: {progress:.0f}%")
        time.sleep(0.1)

    writer("처리 완료!")

    return {"result": total}

In [55]:
graph.add_node("process", process_node)
graph.add_edge(START, "process")
graph.add_edge("process", END)

app = graph.compile()

In [56]:
for chunk in app.stream(
    {"data": [1, 2, 3, 4, 5], "result": 0},
    stream_mode="custom"
):
    print(f"알림: {chunk}")

알림: 처리 시작...
알림: 진행률: 20%
알림: 진행률: 40%
알림: 진행률: 60%
알림: 진행률: 80%
알림: 진행률: 100%
알림: 처리 완료!


<br>

#### 노드에서 커스텀 데이터 스트리밍

<br>

#### 다단계 파일 처리 예시


In [57]:
from langgraph.graph import StateGraph, START, END
from langgraph.config import get_stream_writer
from typing_extensions import TypedDict
import time

In [59]:
class FileProcessState(TypedDict):
    file_path: str
    status: str

graph = StateGraph(FileProcessState)

In [64]:
def download_file(state: FileProcessState):
    writer = get_stream_writer()

    writer({"stage": "download", "status": "시작", "progress": 0})

    for i in range(1, 6):
        time.sleep(0.2)
        writer({
            "stage": "download",
            "status": "진행 중",
            "progress": i * 20
        })

    writer({"stage": "download", "status": "완료", "progress": 100})

    return {"status": "다운로드 완료"}

In [ ]:
def process_file(state: FileProcessState):
    writer = get_stream_writer()

    writer({"stage": "process", "status": "시작", "progress": 0})

    for i in range(1, 6):
        time.sleep(0.2)
        writer({
            "stage": "process",
            "status": "처리 중",
            "progress": i * 20
        })

    writer({"stage": "process", "status": "완료", "progress": 100})

    return {"status": "처리 완료"}

In [62]:
graph.add_node("download", download_file)
graph.add_node("process", process_file)

graph.add_edge(START, "download")
graph.add_edge("download", "process")
graph.add_edge("process", END)

app = graph.compile()

In [63]:
for chunk in app.stream(
    {"file_path": "/data/file.txt", "status": "대기 중"},
    stream_mode="custom"
):
    stage = chunk.get("stage", "unknown")
    status = chunk.get("status", "")
    progress = chunk.get("progress", 0)
    print(f"[{stage.upper()}] {status} - {progress}%")

[DOWNLOAD] 시작 - 0%
[DOWNLOAD] 진행 중 - 20%
[DOWNLOAD] 진행 중 - 40%
[DOWNLOAD] 진행 중 - 60%
[DOWNLOAD] 진행 중 - 80%
[DOWNLOAD] 진행 중 - 100%
[DOWNLOAD] 완료 - 100%
[PROCESS] 시작 - 0%
[PROCESS] 처리 중 - 20%
[PROCESS] 처리 중 - 40%
[PROCESS] 처리 중 - 60%
[PROCESS] 처리 중 - 80%
[PROCESS] 처리 중 - 100%
[PROCESS] 완료 - 100%


<br>

#### 도구에서 커스텀 데이터 스트리밍

In [71]:
from langchain.tools import tool
from langgraph.config import get_stream_writer
from langgraph.prebuilt import create_react_agent
import time

In [72]:
@tool
def search_database(query: str) -> str:
    """데이터베이스를 검색합니다."""
    writer = get_stream_writer()

    writer(f"검색 시작: '{query}'")

    results = []
    for i in range(3):
        time.sleep(0.3)
        result = f"결과 {i+1}: {query}에 대한 데이터"
        results.append(result)
        writer(f"발견: {result}")

    writer(f"검색 완료: 총 {len(results)}개 결과")

    return "\n".join(results)

In [73]:
agent = create_react_agent(
    model="openai:gpt-4.1-mini",
    tools=[search_database],
)

C:\Users\user\AppData\Local\Temp\ipykernel_43316\996666056.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [74]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "LangGraph 검색해줘"}]},
    stream_mode="custom"
):
    print(f"검색: {chunk}")

검색: 검색 시작: 'LangGraph'
검색: 발견: 결과 1: LangGraph에 대한 데이터
검색: 발견: 결과 2: LangGraph에 대한 데이터
검색: 발견: 결과 3: LangGraph에 대한 데이터
검색: 검색 완료: 총 3개 결과


<br>

#### 다중 모드와 함께 사용 예시
- `custom` 모드를 다른 모드와 결합하여 사용

In [75]:
from langgraph.graph import StateGraph, START, END
from langgraph.config import get_stream_writer
from typing_extensions import TypedDict

In [76]:
class AnalysisState(TypedDict):
    text: str
    word_count: int
    sentiment: str

graph = StateGraph(AnalysisState)

In [77]:
def analyze_text(state: AnalysisState):
    writer = get_stream_writer()

    writer({"type": "log", "message": "분석 시작"})

    word_count = len(state["text"].split())
    writer({"type": "progress", "step": "단어 수 계산", "value": word_count})

    sentiment = "긍정적" if "좋" in state["text"] else "중립"
    writer({"type": "progress", "step": "감정 분석", "value": sentiment})

    writer({"type": "log", "message": "분석 완료"})

    return {"word_count": word_count, "sentiment": sentiment}

In [78]:
graph.add_node("analyze", analyze_text)
graph.add_edge(START, "analyze")
graph.add_edge("analyze", END)

app = graph.compile()

In [79]:
for mode, chunk in app.stream(
    {"text": "LangGraph는 정말 좋은 프레임워크입니다", "word_count": 0, "sentiment": ""},
    stream_mode=["values", "custom"]
):
    if mode == "custom":
        msg_type = chunk.get("type", "unknown")
        if msg_type == "log":
            print(f"로그: {chunk['message']}")
        elif msg_type == "progress":
            print(f"진행: {chunk['step']} -> {chunk['value']}")
    elif mode == "values":
        print(f"상태: {chunk}")

상태: {'text': 'LangGraph는 정말 좋은 프레임워크입니다', 'word_count': 0, 'sentiment': ''}
로그: 분석 시작
진행: 단어 수 계산 -> 4
진행: 감정 분석 -> 긍정적
로그: 분석 완료
상태: {'text': 'LangGraph는 정말 좋은 프레임워크입니다', 'word_count': 4, 'sentiment': '긍정적'}


<br>

### 진행 표시줄

In [80]:
from tqdm import tqdm
from langgraph.graph import StateGraph, START, END
from langgraph.config import get_stream_writer
from typing_extensions import TypedDict
import time

In [81]:
class BatchProcessState(TypedDict):
    items: list[str]
    processed: list[str]

In [82]:
def process_batch(state: BatchProcessState):
    writer = get_stream_writer()

    processed = []
    total = len(state["items"])

    for i, item in enumerate(state["items"]):
        time.sleep(0.1)
        processed.append(f"처리됨: {item}")

        writer({
            "current": i + 1,
            "total": total,
            "item": item
        })

    return {"processed": processed}

In [83]:
graph = StateGraph(BatchProcessState)
graph.add_node("process", process_batch)
graph.add_edge(START, "process")
graph.add_edge("process", END)

app = graph.compile()

In [84]:
items = [f"항목{i}" for i in range(1, 11)]

with tqdm(total=len(items), desc="배치 처리") as pbar:
    for chunk in app.stream(
        {"items": items, "processed": []},
        stream_mode="custom"
    ):
        current = chunk.get("current", 0)
        item = chunk.get("item", "")
        pbar.set_description(f"처리 중: {item}")
        pbar.update(1)

처리 중: 항목10: 100%|██████████| 10/10 [00:01<00:00,  9.89it/s]


<br>

#### 로깅 시스템 예시

In [85]:
import logging
from datetime import datetime
from langgraph.graph import StateGraph, START, END
from langgraph.config import get_stream_writer
from typing_extensions import TypedDict
import time

In [86]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

In [87]:
class LoggingState(TypedDict):
    task: str
    result: str

In [88]:
def task_with_logging(state: LoggingState):
    writer = get_stream_writer()

    writer({"level": "INFO", "message": f"작업 시작: {state['task']}"})

    try:
        time.sleep(0.5)
        writer({"level": "DEBUG", "message": "중간 처리 중..."})

        result = f"{state['task']} 완료"
        writer({"level": "INFO", "message": f"작업 성공: {result}"})

        return {"result": result}

    except Exception as e:
        writer({"level": "ERROR", "message": f"작업 실패: {str(e)}"})
        raise

In [89]:
graph = StateGraph(LoggingState)
graph.add_node("task", task_with_logging)
graph.add_edge(START, "task")
graph.add_edge("task", END)

app = graph.compile()

In [90]:
for chunk in app.stream(
    {"task": "데이터 분석", "result": ""},
    stream_mode="custom"
):
    level = chunk.get("level", "INFO")
    message = chunk.get("message", "")

    if level == "INFO":
        logger.info(message)
    elif level == "DEBUG":
        logger.debug(message)
    elif level == "ERROR":
        logger.error(message)

2026-06-09 18:13:00,879 - INFO - 작업 시작: 데이터 분석
2026-06-09 18:13:01,379 - INFO - 작업 성공: 데이터 분석 완료


<br>

<hr>

<br>